# Reproducing Gillespie et al. (2021) from dandiset 000115

Paper. Gillespie AK, Astudillo Maya DA, Denovellis EL, Liu DF, Kastner DB, Coulter ME,
Roumis DK, Eden UT, Frank LM (2021). *Neuron* 109, 3149-3163.
[doi:10.1016/j.neuron.2021.07.029](https://doi.org/10.1016/j.neuron.2021.07.029). The claims file
was written from the accepted author manuscript in PubMed Central (PMC8497431) with its
Supplemental Information, and from the paper's code (LorenFrankLab/Gillespie_Neuron_2021, commit
1be6a6a).

Dandiset. [DANDI:000115](https://dandiarchive.org/dandiset/000115/0.210914.1732), version
0.210914.1732 (doi:10.48324/dandi.000115/0.210914.1732). The paper cites no version, and its
code's README links the draft. 0.210914.1732 is the only published version, and it is pinned.
The draft (modified 2024-02-22) has the same 57 paths, but 55 of its files were rewritten in
January 2022 (NWB 2.4.0). The data check below compares one of them: the DIO timestamps agree to
within a microsecond, but the draft's position timestamps are no longer on the DIO clock, while
the published version's are. The published version therefore fits the analysis better as well as
being fixed.

What the dandiset holds. One NWB file per recording day for each of four rats (jaq,
roquefort, despereaux, montague): the raw 30 kHz broadband of 128 channels, port beam breaks,
lights and pumps, head position, and the StateScript log of each epoch, which records every port
entry, reward, timeout and the current goal arm. It holds no spike times, LFP, ripple times,
linearised position or decoder output. Every replay and ripple claim would have to start from
the raw broadband, about 9.1 TB in all.

Questions answered. Two questions on fig1d-sessions-blocks were answered by a reader and the
claims file updated. Which jaq sessions are "sessions 1 and 3": the paper does not say, and the
criterion now allows (a) the first and third run sessions in order of recording and (b) the two
sessions the Methods describe, with (b) deciding when they differ. Which trial closes a block:
the first valid trial after each goal change, whatever arm it visits, as the paper's code does.
Both claims are re-run below under these rules. After the first verification round, the data
check adds a per-session table, the arms entered in the DIO, and trial counts against the
legends; fig1ef-reward-rate adds the per-trial values; and the behavioural half of
fig6b-visits-decay is computed.

Claims. 33 claims. Two can be tested here, both on behaviour. The other 31 need ripples or the
clusterless decoder run on the raw data.

| claim | tested here |
|---|---|
| fig1d-sessions-blocks | tested |
| fig1ef-reward-rate | tested |
| figs1c-swr-reward-consumption | blocked, too_large (ripple detection on raw broadband) |
| figs3abc-decode-accuracy, figs3df-goal-arm-no-remapping, figs3g-continuous-fraction, likelihood-decode-same-segment | blocked, too_large (clusterless decoding) |
| fig2f-remote-fraction, fig3a-box-majority, fig3b-search-prevgoal-enriched, fig3b-search-past-depleted, fig3b-search-future-inconsistent, fig3b-search-other, fig3c-repeat-prevgoal | blocked, too_large |
| fig3d-glm-search, fig3e-glm-repeat, fig4ab-current-goal-late, fig4c-current-goal-reward-correlation | blocked, too_large |
| fig5a-future-replay-rate, fig5b-future-correct-error, fig5c-current-goal-correct-error, figs7-early-late-correct-error, figs7e-remote-rate-error, fig5d-glm-predict-chance | blocked, too_large |
| fig6b-visits-decay | blocked, too_large (its behavioural half is computed below) |
| pg-replay-visits-correlation, fig5e-pg-replay-no-avoidance, fig6a-pg-replay-persists, fig6c-pg-replay-two-blocks, fig7-recency, fig8-outer-prevgoal, figs8-outbound-final-events, robustness-controls | blocked, too_large |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import replaytask as rt

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = rt.list_assets()
days = rt.load_all(assets)
print(f"loaded {len(days)} files of dandiset {rt.DANDISET_ID} version {rt.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
files = pd.DataFrame([{"subject": d["subject"], "day": d["day"], "session_id": d["session_id"],
                       "date": d["session_start"][:10], "epochs": " ".join(d["tags"]),
                       "run_epochs": len(d["runs"]), "GB": round(d["bytes"] / 1e9, 1),
                       "channels": d["n_channels"], "samples": d["n_samples"],
                       "electrode_location": ",".join(d["electrode_locations"])} for d in days])
files = files.sort_values(["subject", "day"]).reset_index(drop=True)
print("Files, run epochs and size per subject:")
print(files.groupby("subject").agg(files=("day", "size"), run_epochs=("run_epochs", "sum"),
                                   TB=("GB", lambda x: round(x.sum() / 1000, 2))).loc[rt.SUBJECTS].to_string())
print("Total size (TB):", round(files.GB.sum() / 1000, 2), "; assets:", len(assets))
print("\nTask names of the run epochs:", sorted({v[0] + " / " + v[1] for d in days for v in d["tasks"].values() if v[0] != "Sleep"}))
print("Electrode 'location' values over all files:", sorted(set(files.electrode_location)))
print("Channels per file:", sorted(set(files.channels)))
print("\n", files.to_string())

The paper used 24, 32, 23 and 33 run sessions after excluding two of subject 1 (26, 32, 23 and
33 before). The files hold 27, 32, 24 and 33 run epochs, all with the task
"Eight arm flexible spatial task / goal_delay"; jaq and despereaux have one more than the paper
reports. The electrode table gives no location for any channel ("None"), so the CA1 cell-layer
tetrodes and the corpus callosum reference tetrode the Methods name cannot be identified from the
files.

In [ ]:
runs = []
for d in days:
    for r in d["runs"]:
        tr = r["trials"]
        v = tr[~tr.lockout]
        clk = rt.statescript_to_nwb_clock(r)
        runs.append({"subject": d["subject"], "day": d["day"], "epoch": r["epoch_index"], "tag": r["epoch_tag"],
                     "minutes": round(r["duration_min"], 1), "trials": len(tr), "lockout": int(tr.lockout.sum()),
                     "valid": len(v), "rewards_log": int(v.rewarded.sum()), "reward_lines": r["n_outer_reward_lines"],
                     "pumps_dio": len(r["dio_pump_t"]), "arms_visited": v.outer.nunique(),
                     "forageNum": r["forageNum"], "outerReps_setting": r["outerReps_setting"],
                     "outerreps_printed": r["outerreps_printed"],
                     "clock_slope": None if clk is None else clk[0],
                     "clock_median_ms": None if clk is None else round(clk[2], 1),
                     "clock_max_ms": None if clk is None else round(clk[3], 1),
                     "unmatched_pumps": None if clk is None else clk[5],
                     "pos_fs": round(r.get("pos_fs", np.nan), 2), "pos_nan_t": r.get("pos_nan_t"),
                     "pos_nonincreasing": r.get("pos_nonincreasing"), "pos_unit": r.get("pos_unit"),
                     "pos_x_range_m": round(float(np.ptp(r["pos_xy"][:, 0])), 3),
                     "pos_y_range_m": round(float(np.ptp(r["pos_xy"][:, 1])), 3),
                     "sc_hash": r["sc_hash"], "run": r})
runs = pd.DataFrame(runs).sort_values(["subject", "day", "epoch"]).reset_index(drop=True)
runs["session"] = runs.groupby("subject").cumcount() + 1
show = runs.drop(columns=["run", "sc_hash"])
print(show.to_string())

In [ ]:
print("Run epochs per subject:", runs.groupby("subject").size().loc[rt.SUBJECTS].to_dict())
print("Epoch length (min): median", runs.minutes.median(), "range", runs.minutes.min(), "to", runs.minutes.max(),
      "(Methods: 45 to 90 min)")
print("forageNum values (forage assist when above 1):", runs.forageNum.fillna("not in log header").value_counts().to_dict())
print("outerReps settings:", runs.outerReps_setting.fillna("not in log header").value_counts().to_dict())
print("\nValid (non-timeout) trials per subject:", runs.groupby("subject").valid.sum().loc[rt.SUBJECTS].to_dict())
print("Timeout trials per subject:", runs.groupby("subject").lockout.sum().loc[rt.SUBJECTS].to_dict())
print("\nLogged rewards equal 'outer reward' lines in", int((runs.rewards_log == runs.reward_lines).sum()), "of", len(runs), "epochs")
print("Logged rewards equal DIO pump onsets in", int((runs.rewards_log == runs.pumps_dio).sum()), "of", len(runs), "epochs")
print(runs.loc[runs.rewards_log != runs.pumps_dio, ["subject", "day", "epoch", "rewards_log", "pumps_dio", "unmatched_pumps"]].to_string())
# Rewarded only when the visited arm is the goal the log printed for that trial
allv = pd.concat([r["trials"][~r["trials"].lockout] for r in runs.run])
print("\nRewarded exactly when the visited arm is the printed goal:", bool((allv.rewarded == (allv.outer == allv.goal)).all()),
      f"({len(allv)} trials)")
# Rewards per completed block against the programmed number
comp = []
for _, row in runs.iterrows():
    lab = rt.label_task_phase(row.run["trials"])
    for k in lab.block.unique():
        b = lab[lab.block == k]
        if b.final.any():
            comp.append({"subject": row.subject, "rewards": int(b.rewarded[~b.final].sum())})
comp = pd.DataFrame(comp)
print("\nRewards per completed block (Methods: 15, then 10, then 4 to 12 for two subjects):")
print(pd.crosstab(comp.subject, comp.rewards).loc[rt.SUBJECTS].to_string())

Trials are rebuilt from the StateScript log (see `replaytask.py`). In every one of the 116 run
epochs, the logged rewards equal the log's own `outer reward` lines, and a trial is rewarded
exactly when the visited arm is the goal printed for that trial. The logged rewards equal the
reward pump onsets of the DIO channels in 115 epochs. In jaq day 6 epoch 4 the DIO has one more
pump (arm 8) 226 s after the last logged reward, with no log line, so the log appears to stop
before the epoch ends. The number of rewards per completed block is a programmed value (15 and 10, and 4 to 12 for
despereaux and montague, as the Methods and Supplementary Figure 1A say) in all blocks but two,
jaq with 9 and montague with 3; the per-session check below explains both. forageNum is 1 in the
111 logs whose header holds it; five logs carry no Python header, and the per-session check
below tests them for forage assist from the log lines.

In [ ]:
print("StateScript clock to NWB clock, fit on rewarded entries against DIO pump onsets:")
print("slope range:", runs.clock_slope.min(), "to", runs.clock_slope.max())
print("median residual per epoch (ms): median", runs.clock_median_ms.median(), "max", runs.clock_median_ms.max())
print("largest residual per epoch (ms): median", runs.clock_max_ms.median(), "; epochs above 50 ms:")
print(runs.loc[runs.clock_max_ms > 50, ["subject", "day", "epoch", "clock_median_ms", "clock_max_ms"]].to_string())
bad = runs.loc[runs.clock_max_ms > 50].iloc[0].run
tr = bad["trials"]
rt_s = tr[tr.rewarded & ~tr.lockout].t_ms.to_numpy() / 1000
print("Interval differences between consecutive rewards, log minus DIO (s), in that epoch:",
      np.round(np.diff(rt_s) - np.diff(bad["dio_pump_t"]), 2)[:8], "...")

# Position against DIO beam breaks: decode the poked arm from position at the beam break
acc = []
for _, row in runs.iterrows():
    a0, n = rt.arm_from_position(row.run, 0.0)
    a1, _ = rt.arm_from_position(row.run, -20.0)
    a2, _ = rt.arm_from_position(row.run, 20.0)
    acc.append({"subject": row.subject, "acc_0s": a0, "acc_minus20s": a1, "acc_plus20s": a2, "entries": n})
acc = pd.DataFrame(acc)
print("\nPoked arm decoded from head position at the DIO beam break (leave-one-out nearest centroid; chance about 0.125):")
print(acc.groupby("subject")[["acc_0s", "acc_minus20s", "acc_plus20s"]].median().loc[rt.SUBJECTS].round(3).to_string())
print("epochs:", len(acc), "; outer beam breaks per epoch: median", acc.entries.median(),
      "; lowest accuracy at 0 s:", round(acc.acc_0s.min(), 3))

The StateScript times map onto the NWB clock with a slope of 1 and a median residual of a few
milliseconds, so the trial table and the DIO events share one clock. In montague day 16 epoch 2
the log's clock steps by about 3.4 s twice against the DIO early in the epoch; the trials are
unaffected because the analysis uses their order, not their times. Head position at the moment
of each outer-port beam break identifies the arm, while position taken 20 s earlier or later
does not, so position and the DIO share a clock. The raw broadband is not used here; its
timestamps are checked only for the span below.

In [ ]:
pos = runs[["subject", "pos_fs", "pos_nan_t", "pos_nonincreasing", "pos_unit", "pos_x_range_m", "pos_y_range_m"]]
print("Position: unit", pos.pos_unit.unique().tolist(), "; sampling rate (Hz)", pos.pos_fs.min(), "to", pos.pos_fs.max(),
      "(Methods: 30 Hz)")
print("x range (m):", pos.pos_x_range_m.min(), "to", pos.pos_x_range_m.max(), "; y range (m):",
      pos.pos_y_range_m.min(), "to", pos.pos_y_range_m.max())
print("Epochs with NaN position timestamps:", int((pos.pos_nan_t > 0).sum()), "of", len(pos), "; NaN timestamps per epoch up to",
      int(pos.pos_nan_t.max()))
print("Epochs with non-increasing position timestamps:", int((pos.pos_nonincreasing > 0).sum()), "; most in one epoch:",
      int(pos.pos_nonincreasing.max()))
print("\nRun epochs whose StateScript log is identical to another's:", int(runs.sc_hash.duplicated().sum()))
ephys = pd.DataFrame([{"subject": d["subject"], "day": d["day"], "samples": d["n_samples"],
                       "span_s": d["ephys_t1"] - d["ephys_t0"], "chunks": str(d["ephys_chunks"]),
                       "conversion": d["ephys_conversion"]} for d in days])
ephys["rate_over_span"] = ephys.samples / ephys.span_s
print("Raw broadband: chunk shapes", ephys.chunks.unique().tolist(), "; conversion to volts", ephys.conversion.unique().tolist())
print("samples / time span (Hz): median", round(ephys.rate_over_span.median()), "range",
      round(ephys.rate_over_span.min()), "to", round(ephys.rate_over_span.max()),
      "(30 kHz with the gaps between epochs inside the span)")
ephys["epoch_seconds"] = [d["epoch_seconds"] for d in days]
ephys["samples_over_epoch_time"] = ephys.samples / ephys.epoch_seconds
print("samples / summed epoch durations (Hz): median", round(ephys.samples_over_epoch_time.median()), "range",
      round(ephys.samples_over_epoch_time.min()), "to", round(ephys.samples_over_epoch_time.max()))
print("Recorded hours of raw broadband at 30 kHz:", round(ephys.samples.sum() / 30000 / 3600, 1))

# Pinned version against the draft, for one file
cmp = rt.compare_with_draft("sub-montague/sub-montague_ses-montague-04_behavior+ecephys.nwb")
pub, dr = cmp[rt.DANDISET_VERSION], cmp["draft"]
print("\nmontague day 4, pinned version against draft:")
for v in (rt.DANDISET_VERSION, "draft"):
    print(f"  {v}: NWB {cmp[v]['nwb_version']}, file created {cmp[v]['file_create_date'][:10]}, "
          f"position timestamps {np.nanmin(cmp[v]['pos_t']):.1f} to {np.nanmax(cmp[v]['pos_t']):.1f} s, "
          f"home beam timestamps {cmp[v]['dio_t'].min():.1f} to {cmp[v]['dio_t'].max():.1f} s")
print("  largest home beam timestamp difference (s):", float(np.max(np.abs(pub['dio_t'] - dr['dio_t']))))
print("  position timestamp difference, draft minus pinned (s): median",
      float(np.nanmedian(dr['pos_t'] - pub['pos_t'])), "; position values identical:", cmp["pos_data_equal"])

Position is stated in metres and spans about 1.1 to 1.3 m on each axis, which fits an
eight-arm maze; it is sampled at about 33 Hz, not the 30 Hz of the Methods. 110 of 116 epochs end
with 1 to 4 NaN timestamps, and two have non-increasing ones; these samples are dropped here. No
two run epochs share a StateScript log, so no recording was deposited twice. The raw broadband
has 30,000 samples per second of summed epoch time, as the Methods state. In the draft, the
position timestamps of the file compared are shifted by about -1.59e9 s from the DIO clock, with
the same position values; the pinned version is used throughout.

Sessions 1 and 3 of subject 1. The Methods exclude "sessions 1 and 3 of subject 1"
for decoding problems: the first had fewer than 75% of samples decoded on the same segment, and
in the second "not all arms were visited". The files do not mark these sessions, and the
answered question says the paper does not number them. The criterion allows two readings.
(a) The first and third run sessions in order of recording, day 6 epochs 2 and 5 (all eight arms
are visited in both). (b) The two sessions the Methods describe. The table below shows that the
only jaq session with fewer than eight arms visited is day 6 epoch 4, so it is the second
excluded session under (b). The first cannot be identified here: it needs the decoder, which is
blocked (see the claims not tested). Reading (b) is therefore run with day 6 epoch 4 and each of
the other 26 jaq sessions in turn as the first; "b: day 6 epochs 2 and 4" below takes the
earliest. With no exclusion, (c) is shown as a variant, not a reading, since the criterion
excludes two sessions.

In [ ]:
jaq = runs[runs.subject == "jaq"][["session", "day", "epoch", "arms_visited", "valid", "rewards_log"]]
print(jaq.head(6).to_string(index=False))
print("jaq sessions with fewer than eight arms visited:",
      jaq.loc[jaq.arms_visited < 8, ["day", "epoch", "arms_visited"]].to_dict("records"))
NOT_ALL_ARMS = tuple(jaq.loc[jaq.arms_visited < 8, ["day", "epoch"]].iloc[0])
B_FIRST_CHOICES = [(d, e) for d, e in zip(jaq.day, jaq.epoch) if (d, e) != NOT_ALL_ARMS]
EXCLUDE = {"a: sessions 1 and 3 in order": [(6, 2), (6, 5)],
           "b: day 6 epochs 2 and 4": [B_FIRST_CHOICES[0], NOT_ALL_ARMS],
           "c, variant: none": []}
print("Reading (b): second session", NOT_ALL_ARMS, ";", len(B_FIRST_CHOICES), "candidates for the first")

def session_set(reading=None, exclude=None):
    ex = EXCLUDE[reading] if exclude is None else exclude
    keep = ~((runs.subject == "jaq") & runs.apply(lambda r: (r.day, r.epoch) in ex, axis=1))
    return runs[keep]

# Streaming cost of the raw broadband, measured
url = assets.url[assets.path.str.contains("jaq-10")].iloc[0]
secs, nbytes = rt.time_raw_read(url)
rate = nbytes / secs
total = files.GB.sum() * 1e9
print(f"\nRead {nbytes / 1e6:.0f} MB of raw broadband (300000 samples x 128 channels) in {secs:.1f} s, {rate / 1e6:.1f} MB/s")
print(f"At that rate the dandiset's {total / 1e12:.1f} TB would take {total / rate / 3600:.0f} h; "
      f"one 90-min run epoch ({90 * 60 * 30000 * 128 * 2 / 1e9:.0f} GB) {90 * 60 * 30000 * 128 * 2 / rate / 60:.0f} min")

The raw broadband is stored one channel per chunk of 212,978 to 329,259 samples (7 to 11 s),
so any time window must fetch every channel's chunk. The rate measured above puts streaming the
run epochs alone (116 epochs of about 41 GB) far beyond the registry's 30-minute limit, before
any filtering, spike detection or decoding. This is why the 31 replay and ripple claims are
blocked as too_large.

### Per-session check

Added after the first verification round. The table lists every run epoch with its log header,
forage-assist lines, blocks, rewards per block, and the arms entered in the log's valid trials and
in the DIO beam breaks.

In [ ]:
# Per-session table: header, forage assist, blocks and rewards per block, arms visited in the log and in the DIO
sess = []
for _, row in runs.iterrows():
    r = row.run
    lab = rt.label_task_phase(r["trials"])
    per_block = [int(lab.rewarded[(lab.block == k) & ~lab.final].sum()) for k in sorted(lab.block.unique())]
    closed = [k for k in sorted(lab.block.unique()) if lab.final[lab.block == k].any()]
    sess.append({"subject": row.subject, "day": row.day, "epoch": row.epoch, "session": row.session,
                 "minutes": row.minutes, "header": r["forageNum"] is not None, "forageNum": r["forageNum"],
                 "multi_arm_goal_lines": r["n_multi_goal_lines"], "outerreps_printed": r["outerreps_printed"],
                 "valid": row.valid, "rewards": row.rewards_log, "full_blocks": len(rt.full_blocks(lab)),
                 "rewards_per_block": " ".join(str(x) + ("" if k in closed else "o") for k, x in zip(sorted(lab.block.unique()), per_block)),
                 "arms_log": row.arms_visited, "arms_dio": len(np.unique(r["dio_beam_arms"]))})
sess = pd.DataFrame(sess)
print("Per run epoch ('o' marks the open last block of the epoch):")
print(sess.to_string(index=False))
print("\nEpochs whose log has no Python header:", sess.loc[~sess.header, ["subject", "day", "epoch"]].to_dict("records"))
print("Epochs with a CURRENTGOAL line naming more than one arm (forage assist):", int((sess.multi_arm_goal_lines > 0).sum()),
      "of", len(sess))
print("Epochs shorter than 45 min:", sess.loc[sess.minutes < 45, ["subject", "day", "epoch", "minutes", "full_blocks"]].to_dict("records"))
print("Epochs where the DIO beam breaks reach a different number of arms than the log:",
      sess.loc[sess.arms_log != sess.arms_dio, ["subject", "day", "epoch", "arms_log", "arms_dio"]].to_dict("records"))

# Completed blocks whose reward count is not a programmed value: is the missing reward on the closing trial of the block before?
odd = []
for _, row in runs.iterrows():
    lab = rt.label_task_phase(row.run["trials"])
    for k in sorted(lab.block.unique()):
        b = lab[lab.block == k]
        n = int(b.rewarded[~b.final].sum())
        programmed = {15, 10} if row.subject in ("jaq", "roquefort") else set(range(4, 13)) | {15}
        if b.final.any() and n not in programmed:
            prev_close = lab[(lab.block == k - 1) & lab.final]
            odd.append({"subject": row.subject, "day": row.day, "epoch": row.epoch, "block": k, "goal": b.block_goal.iloc[0],
                        "rewards_in_block": n, "closing_trial_of_previous_block": f"arm {prev_close.outer.iloc[0]}, rewarded {bool(prev_close.rewarded.iloc[0])}"
                        if len(prev_close) else "none", "outerreps_printed": row.run["outerreps_printed"]})
print("\nCompleted blocks with a reward count outside the programmed values:")
print(pd.DataFrame(odd).to_string(index=False))

# jaq day 6 epoch 4: arms entered according to the DIO, before and after the last logged trial
r = runs[(runs.subject == "jaq") & (runs.day == NOT_ALL_ARMS[0]) & (runs.epoch == NOT_ALL_ARMS[1])].run.iloc[0]
a, b = rt.statescript_to_nwb_clock(r)[:2]
t_last = a * r["trials"].t_ms.max() / 1000 + b
before = r["dio_beam_t"] <= t_last + 1
print(f"\njaq day {NOT_ALL_ARMS[0]} epoch {NOT_ALL_ARMS[1]}: last logged trial at {t_last - r['start']:.0f} s of {r['stop'] - r['start']:.0f} s")
print("  arms visited in the log:", sorted(r["trials"][~r["trials"].lockout].outer.unique().tolist()))
print("  arms with a DIO beam break up to the last logged trial:", sorted(np.unique(r["dio_beam_arms"][before]).tolist()))
print("  DIO beam breaks after it, by arm:", pd.Series(r["dio_beam_arms"][~before]).value_counts().sort_index().to_dict())
print("  DIO pumps after it (s into the epoch, arm):", [(round(t - r["start"]), int(x)) for t, x in zip(r["dio_pump_t"], r["dio_pump_arms"]) if t > t_last + 1])


With forageNum above 1 the observer code in the log headers puts several arms in the goal until
one is found, and prints them all in the `CURRENTGOAL IS [...]` line. No log, including the five
without a header, has such a line, so no epoch uses forage assist. The two completed blocks whose
reward count is not a programmed value (jaq day 12 epoch 2 with 9, montague day 14 epoch 4 with 3)
both follow a block whose closing trial was a rewarded visit to the new goal. Under the closing-trial
rule that reward stays in the old block, so the new block shows one reward fewer than the
outerreps printed (10 and 4). The DIO confirms that in jaq day 6 epoch 4 arm 6 was never entered,
including after the last logged trial, where there are only arm 8 entries and the extra pump. It is
the only jaq session with fewer than eight arms, in the log and in the DIO. In 12 epochs of the other
subjects the DIO reaches an arm that no valid trial of the log visits.

Which run epochs the paper did not count cannot be settled from the files: jaq and despereaux have
one more each. Possible ones are jaq day 7 epoch 2 (6 rewards in 74 valid trials, one open block,
no outerreps printed) and the epochs shorter than the Methods' 45 minutes, listed above.
fig1d-sessions-blocks shows the counts with each jaq and despereaux epoch left out in turn.

In [ ]:
# Trial counts per subject against the counts in the figure legends (not a claim; for comparison only)
LEGEND = pd.DataFrame({"search_GLM_3D": [711, 848, 725, 1065], "search_5A": [534, 617, 570, 879],
                       "repeat_5A": [924, 1019, 894, 1302], "correct_5B": [923, 1007, 847, 1340],
                       "error_5B": [424, 654, 307, 390]}, index=rt.SUBJECTS)
cnt = []
for s in rt.SUBJECTS:
    L = pd.concat([rt.label_task_phase(r["trials"]) for r in session_set("a: sessions 1 and 3 in order").query("subject == @s").run])
    frac = np.round(L.phase % 1, 1)
    rep = L.phase > 1
    cnt.append({"subject": s, "valid": len(L), "search": int((L.phase <= 1).sum()),
                "search_after_block1": int(((L.phase <= 1) & (L.block > 1)).sum()),
                "repeat": int(rep.sum()), "repeat_after_block1": int((rep & (L.block > 1)).sum()),
                "correct_repeat": int((rep & ((frac == 0) | (frac == 0.9))).sum()),
                "error_repeat": int((rep & frac.isin([0.3, 0.5, 0.8])).sum())})
cnt = pd.DataFrame(cnt).set_index("subject")
print("Trial counts under session reading (a), against the legend counts:")
print(pd.concat([cnt, LEGEND], axis=1).to_string())
pairs = {"search / search_GLM_3D": ("search", "search_GLM_3D"), "search_after_block1 / search_5A": ("search_after_block1", "search_5A"),
         "repeat_after_block1 / repeat_5A": ("repeat_after_block1", "repeat_5A"), "correct_repeat / correct_5B": ("correct_repeat", "correct_5B"),
         "error_repeat / error_5B": ("error_repeat", "error_5B")}
both = pd.concat([cnt, LEGEND], axis=1)
print("\nRatio of the count here to the legend's:")
print(pd.DataFrame({k: (both[a] / both[b]).round(3) for k, (a, b) in pairs.items()}).to_string())


These counts are a comparison, not a claim: the legends do not say exactly which trials each count
covers. All search trials are within 5% of the Figure 3D counts (montague equal), search and repeat
trials after the first block within 8% of Figure 5A's, and correct repeat trials within 4% of Figure
5B's. Error trials are 11 to 24% more than Figure 5B's, so the error labelling, or the selection
behind it, is not the paper's. Only the replay claims on error trials use it, and all are blocked.

## fig1d-sessions-blocks

> The analysis includes 24, 32, 23 and 33 behavioural sessions and 89, 85, 105 and 166 trial
> blocks for the four subjects.

Criterion: sessions are the full-task sessions less sessions 1 and 3 of subject 1, under reading
(a) or (b) (above), with (b) deciding when they differ; blocks are those with a trial of task
phase >= 4 (`any(ctphase>=4)`, dfs_ripcontent.m part 2), with the first valid trial after each
goal change in the old block (`replaytask.FINAL_TRIAL`). Reproduced when all eight counts are
within tolerance. The other closing-trial rule ("old_goal_visit", a reading before the answer)
is shown only to say what the answer changed.

In [ ]:
EXPECTED_SESSIONS = {"jaq": 24, "roquefort": 32, "despereaux": 23, "montague": 33}
EXPECTED_BLOCKS = {"jaq": 89, "roquefort": 85, "despereaux": 105, "montague": 166}
for final in rt.FINAL_TRIAL_READINGS:
    runs["blocks_" + final] = [len(rt.full_blocks(rt.label_task_phase(r["trials"], final))) for r in runs.run]

def count_1d(ss, final):
    out = []
    for s in rt.SUBJECTS:
        nb = ss.loc[ss.subject == s, "blocks_" + final].to_numpy()
        out.append({"subject": s, "sessions": len(nb), "expected_sessions": EXPECTED_SESSIONS[s],
                    "blocks": int(nb.sum()), "expected_blocks": EXPECTED_BLOCKS[s],
                    "blocks_per_session_median": np.median(nb), "blocks_per_session_range": f"{nb.min()}-{nb.max()}"})
    out = pd.DataFrame(out)
    out["sessions_ok"] = (out.sessions - out.expected_sessions).abs() <= 0.1 * out.expected_sessions
    out["blocks_ok"] = (out.blocks - out.expected_blocks).abs() <= 0.1 * out.expected_blocks
    return out

rows = []
for reading in EXCLUDE:
    for final in rt.FINAL_TRIAL_READINGS:
        rows.append(count_1d(session_set(reading), final).assign(sessions_reading=reading, final_trial=final))
res1d = pd.concat(rows)[["sessions_reading", "final_trial", "subject", "sessions", "expected_sessions", "blocks",
                         "expected_blocks", "blocks_per_session_median", "blocks_per_session_range", "sessions_ok", "blocks_ok"]]
print(res1d.to_string(index=False))
print("\nAll eight counts within tolerance, per reading:")
print(res1d.groupby(["sessions_reading", "final_trial"]).apply(lambda g: bool(g.sessions_ok.all() and g.blocks_ok.all())).to_string())

# Reading (b) with every candidate for the session decoded on the same segment less than 75% of the time
b_all = []
for first in B_FIRST_CHOICES:
    g = count_1d(session_set(exclude=[first, NOT_ALL_ARMS]), rt.FINAL_TRIAL)
    jq = g[g.subject == "jaq"].iloc[0]
    b_all.append({"first": first, "jaq_sessions": jq.sessions, "jaq_blocks": jq.blocks,
                  "all_eight_ok": bool(g.sessions_ok.all() and g.blocks_ok.all())})
b_all = pd.DataFrame(b_all)
print(f"\nReading (b), {rt.FINAL_TRIAL}, over the {len(b_all)} candidates for the first excluded jaq session:")
print("  jaq sessions:", sorted(b_all.jaq_sessions.unique().tolist()), "; jaq blocks", b_all.jaq_blocks.min(), "to",
      b_all.jaq_blocks.max(), f"(tolerance {EXPECTED_BLOCKS['jaq'] * 0.9:.1f} to {EXPECTED_BLOCKS['jaq'] * 1.1:.1f})")
print("  candidates with all eight counts within tolerance:", int(b_all.all_eight_ok.sum()), "of", len(b_all))

# What the closing-trial rule does at each goal change
close = []
for _, row in runs.iterrows():
    lab = rt.label_task_phase(row.run["trials"], rt.FINAL_TRIAL)
    for i in np.where(lab.final)[0]:
        kind = ("old goal" if lab.outer[i] == lab.block_goal[i]
                else "new goal, rewarded" if lab.rewarded[i] else "other arm")
        close.append({"subject": row.subject, "closing trial visits": kind})
close = pd.DataFrame(close)
print("\nClosing trials of the blocks (first valid trial after each goal change), by arm visited:")
print(pd.crosstab(close.subject, close["closing trial visits"]).loc[rt.SUBJECTS].to_string())
diff = res1d.pivot_table(index=["sessions_reading", "subject"], columns="final_trial", values="blocks")
print("\nBlocks under the two closing-trial rules (old_goal_visit is the reading before the answer):")
print(diff.loc[[k for k in EXCLUDE if not k.startswith("c")]].to_string())
# Variant: one more jaq or despereaux epoch left out (the files hold one more than the paper for each)
loo = []
for subj in ["jaq", "despereaux"]:
    base = session_set("a: sessions 1 and 3 in order")
    for _, row in base[base.subject == subj].iterrows():
        g = count_1d(base.drop(index=row.name), rt.FINAL_TRIAL)
        sg = g[g.subject == subj].iloc[0]
        loo.append({"subject": subj, "left_out": (row.day, row.epoch), "sessions": sg.sessions, "blocks": sg.blocks,
                    "all_eight_ok": bool(g.sessions_ok.all() and g.blocks_ok.all())})
loo = pd.DataFrame(loo)
print("\nReading (a) with one more epoch left out, per subject:")
print(loo.groupby("subject").agg(sessions=("sessions", "first"), blocks_min=("blocks", "min"), blocks_max=("blocks", "max"),
                                 all_eight_ok=("all_eight_ok", "sum"), candidates=("all_eight_ok", "size")).to_string())


**Result: reproduced.** With the first trial after each goal change closing the old block, all
eight counts are within 10% under reading (a) and under reading (b) for every candidate first
session. Roquefort and montague match the paper's session counts exactly (32 and 33), and their
block counts too (85 and 166). Jaq has 25 sessions against 24, and 88 blocks under reading (a)
against 89. Under reading (b), with day 6 epoch 4 as the session where not all arms were visited,
jaq has 88 to 93 blocks over the 26 candidates for the other session (91 with day 6 epoch 2).
Despereaux has 24 sessions against 23 and 109 blocks against 105. The closing-trial rule settled
by the answer changes only montague, from 167 blocks under the earlier old-goal-visit reading to
166. With no session excluded (variant c), jaq has 27 sessions, outside the tolerance of 24, and
96 blocks, inside that of 89. One run epoch each of jaq and despereaux is in the files but not in
the paper's count, and which one cannot be told; leaving out any one more epoch gives the
paper's session counts (24 and 23), and the block counts stay within tolerance for every choice
(see the variant above).

## fig1ef-reward-rate

> Aligned to the goal change, the outer-arm reward rate starts at chance (0.125) and rises;
> aligned to the first reward at the new goal, it is well above chance, so all subjects keep
> visiting a goal once found.

Criterion: per subject over the blocks with at least four rewards, the final unrewarded trial
dropped; a trial scores 1 when rewarded; binomial S.E.M.; trials 1 to 15. Reproduced when, in all
four subjects, the band on trial 1 of Figure 1E reaches 0.125 or lies below it, the rate rises
over trials 1 to 15, and every point of Figure 1F has its band above 0.125. "Rises" is shown two
ways: trial 15 against trial 1 (non-overlapping bands), and the Spearman correlation of the
rate with trial number. Blocks are those of fig1d-sessions-blocks, closed by the first trial after
each goal change (`replaytask.FINAL_TRIAL`); that closing trial is the one dropped. Sessions
follow readings (a) and (b), and for (b) every candidate first session of jaq is run. The
old-goal-visit rule is shown only to say what the answer changed.

In [ ]:
def curves(reading=None, final=rt.FINAL_TRIAL, exclude=None, subjects=rt.SUBJECTS):
    ss = session_set(reading, exclude)
    out = {}
    for s in subjects:
        S, R = [], []
        for r in ss[ss.subject == s].run:
            a, b = rt.reward_curves(rt.label_task_phase(r["trials"], final))
            S.append(a); R.append(b)
        out[s] = (np.vstack(S), np.vstack(R))
    return out

def summarise_1ef(cv):
    rows = []
    for s, (E, F) in cv.items():
        p, se, n = rt.binomial_curve(E)
        p2, se2, n2 = rt.binomial_curve(F)
        rho, prho = stats.spearmanr(rt.CURVE_TRIALS, p)
        rows.append({"subject": s, "blocks": len(E),
                     "E_trial1": round(p[0], 3), "E_trial1_lower": round(p[0] - se[0], 3),
                     "E_trial15": round(p[14], 3), "E_trial15_lower": round(p[14] - se[14], 3),
                     "E_rho": round(rho, 3), "E_rho_p": prho,
                     "F_min": round(p2.min(), 3), "F_min_lower": round((p2 - se2).min(), 3),
                     "F_n_trial15": int(n2[14])})
    out = pd.DataFrame(rows)
    out["E_starts_at_chance"] = out.E_trial1_lower <= rt.CHANCE
    out["E_rises"] = (out.E_trial15_lower > out.E_trial1 + (out.E_trial1 - out.E_trial1_lower)) & (out.E_rho > 0) & (out.E_rho_p < 0.05)
    out["E_rises_plain"] = out.E_trial15 > out.E_trial1
    out["F_above_chance"] = out.F_min_lower > rt.CHANCE
    return out

ok = lambda g: bool(g.E_starts_at_chance.all() and g.E_rises.all() and g.F_above_chance.all())
res1ef = pd.concat([summarise_1ef(curves(reading, final)).assign(sessions_reading=reading, final_trial=final)
                    for reading in EXCLUDE for final in rt.FINAL_TRIAL_READINGS])
print(res1ef.to_string(index=False))
print("\nAll three conditions in all four subjects, per reading:")
print(res1ef.groupby(["sessions_reading", "final_trial"]).apply(ok).to_string())

# Reading (b): jaq with every candidate for the first excluded session (other subjects do not depend on it)
jb = pd.concat([summarise_1ef(curves(exclude=[first, NOT_ALL_ARMS], subjects=["jaq"])).assign(first=str(first))
                for first in B_FIRST_CHOICES])
print(f"\nReading (b), {rt.FINAL_TRIAL}, jaq over the {len(jb)} candidates for the first excluded session:")
print("  all three conditions hold in", int(sum(ok(jb.iloc[[i]]) for i in range(len(jb)))), "of", len(jb))
for c in ["blocks", "E_trial1", "E_trial1_lower", "E_trial15", "E_trial15_lower", "E_rho", "F_min_lower"]:
    print(f"  {c}: {jb[c].min()} to {jb[c].max()}")
print("  largest Spearman p:", f"{jb.E_rho_p.max():.1e}")
# Per-trial values, reading (a)
cva = curves("a: sessions 1 and 3 in order", rt.FINAL_TRIAL)
per = []
for s, mats in cva.items():
    for fig, mat in zip(["1E", "1F"], mats):
        p, se, n = rt.binomial_curve(mat)
        per += [{"subject": s, "figure": fig, "trial": t, "p": round(p[j], 3), "sem": round(se[j], 3), "n": int(n[j])}
                for j, t in enumerate(rt.CURVE_TRIALS)]
per = pd.DataFrame(per)
print("\nPer-trial reward rate, S.E.M. and blocks, reading (a):")
print(per.pivot(index="trial", columns=["figure", "subject"], values=["p", "sem", "n"]).to_string())


In [ ]:
cv = curves("a: sessions 1 and 3 in order", rt.FINAL_TRIAL)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
cols = ["#1b5c29", "#197b64", "#219fa9", "#7be1bf"]
for s, c in zip(rt.SUBJECTS, cols):
    for ax, mat in zip(axes, cv[s]):
        p, se, n = rt.binomial_curve(mat)
        ax.fill_between(rt.CURVE_TRIALS, p - se, p + se, color=c, alpha=0.3, lw=0)
        ax.plot(rt.CURVE_TRIALS, p, ".-", color=c, label=f"{s} ({len(mat)} blocks)")
for ax, t in zip(axes, ["trials since goal change (1E)", "trials since first reward at new goal (1F)"]):
    ax.axhline(rt.CHANCE, color="k", ls=":")
    ax.set_xlabel(t)
    ax.set_ylim(0, 1)
axes[0].set_ylabel("reward rate")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
lo = min(np.nanmin(rt.binomial_curve(m)[0] - rt.binomial_curve(m)[1]) for s in rt.SUBJECTS for m in cv[s])
hi = max(np.nanmax(rt.binomial_curve(m)[0] + rt.binomial_curve(m)[1]) for s in rt.SUBJECTS for m in cv[s])
print(f"Plotted values (mean +- S.E.M.) span {lo:.3f} to {hi:.3f}; none fall outside the axes (0 to 1).")

**Result: reproduced.** With the first trial after each goal change closing the old block, under
reading (a) and under reading (b) for every candidate first session of jaq, all three conditions
hold in all four subjects. The trial 1 mean of Figure 1E is 0.065 to 0.133, with its lower band at
0.106 or below; jaq (0.068) and roquefort (0.094) start below chance, not at it, which the
criterion's "or lies below" allows. The rate rises to 0.584 to 0.710 by trial 15, with bands clear of trial 1's, and
the Spearman correlation with trial number is 0.943 to 0.993. This test of "rises" is stricter
than the criterion, which only says the rate rises; the plain reading (trial 15 above trial 1)
also holds in every subject. After the first reward at the new
goal, every point of the band is above 0.125, and the lowest lower bound is 0.234 (roquefort).
The closing-trial rule settled by the answer moves no condition. Under the earlier old-goal-visit
reading the same three conditions also hold.

## fig6b-visits-decay (behavioural half only)

> The behavioural bias to the previously rewarded arm decays very quickly after the switch,
> returning to baseline within a few trials.

Criterion: as fig6a-pg-replay-persists for visits (1 if the arm was visited on that trial),
unsmoothed. Per session, for blocks 1 to 8, the goal arm's visits from 40 trials before to 40
after the block's last rewarded trial (trial 0), NaN beyond the session, against the same window
for every arm never a goal in the session; per trial a Wilcoxon rank-sum test of goal against
non-goal, Bonferroni corrected (0.05/80 in the legend, 0.05/81 in the code). Reproduced when, in all
four subjects, the last trial after 0 with a significant goal over non-goal difference is no later
than the bound, (a) +3 or (b) +5, and the replay enrichment of fig6a extends beyond it. The second
part needs replay and is blocked; this section computes the first, which the verifier asked for.
The test is scipy's Mann-Whitney U with the normal approximation and tie correction; scipy's
`ranksums`, without tie correction, is shown as a variant, and so are completed blocks only.

In [ ]:
OFFS = np.arange(-rt.VISIT_WINDOW, rt.VISIT_WINDOW + 1)
N_TESTS = {"0.05/80 (legend)": 80, "0.05/81 (code)": 81}
BOUNDS = {"a: +3": 3, "b: +5": 5}

def visit_sets(ss, completed_only=False):
    out = {}
    for s in rt.SUBJECTS:
        G, N = zip(*[rt.visit_traces(rt.label_task_phase(r["trials"]), completed_only) for r in ss[ss.subject == s].run])
        out[s] = (np.vstack(G), np.vstack(N))
    return out

def trialwise(G, N, test="mannwhitneyu"):
    p = np.full(len(OFFS), np.nan)
    for j in range(len(OFFS)):
        g, n = G[:, j][~np.isnan(G[:, j])], N[:, j][~np.isnan(N[:, j])]
        if len(g) and len(n):
            p[j] = (stats.mannwhitneyu(g, n, alternative="two-sided", method="asymptotic").pvalue if test == "mannwhitneyu"
                    else stats.ranksums(g, n).pvalue)
    return np.nanmean(G, axis=0), np.nanmean(N, axis=0), p

def summarise_6b(vs, test="mannwhitneyu"):
    rows = []
    for s, (G, N) in vs.items():
        gm, nm, p = trialwise(G, N, test)
        for lab, k in N_TESTS.items():
            sig = p < 0.05 / k
            above = OFFS[sig & (gm > nm) & (OFFS > 0)]
            below = OFFS[sig & (gm < nm) & (OFFS > 0)]
            rows.append({"subject": s, "alpha": lab, "goal_traces": len(G), "nongoal_traces": len(N),
                         "last_sig_above_after_0": int(above.max()) if len(above) else None,
                         "sig_above_after_0": " ".join(map(str, above)),
                         "sig_below_after_0": " ".join(map(str, below)),
                         "sig_above_before_0": int(np.sum(sig & (gm > nm) & (OFFS < 0)))})
    out = pd.DataFrame(rows)
    for b, v in BOUNDS.items():
        out["within_" + b] = out.last_sig_above_after_0.fillna(0) <= v
    return out

res6b = []
for reading in EXCLUDE:
    for comp in (False, True):
        res6b.append(summarise_6b(visit_sets(session_set(reading), comp)).assign(
            sessions_reading=reading, blocks="completed only (variant)" if comp else "blocks 1 to 8"))
res6b.append(summarise_6b(visit_sets(session_set("a: sessions 1 and 3 in order")), "ranksums").assign(
    sessions_reading="a: sessions 1 and 3 in order", blocks="blocks 1 to 8, ranksums without tie correction (variant)"))
res6b = pd.concat(res6b)
print(res6b.drop(columns=["sig_above_before_0"]).to_string(index=False))
print("\nBehavioural half holds in all four subjects (last significant goal-above trial after 0 within the bound):")
print(res6b.groupby(["sessions_reading", "blocks", "alpha"])[["within_a: +3", "within_b: +5"]].all().to_string())

# Reading (b): jaq with every candidate for the first excluded session
jb6 = pd.concat([summarise_6b({"jaq": visit_sets(session_set(exclude=[f, NOT_ALL_ARMS]))["jaq"]}).assign(first=str(f))
                 for f in B_FIRST_CHOICES])
print(f"\nReading (b), jaq over the {len(B_FIRST_CHOICES)} candidates: last significant goal-above trial after 0 from",
      jb6.last_sig_above_after_0.min(), "to", jb6.last_sig_above_after_0.max(), "; goal traces",
      jb6.goal_traces.min(), "to", jb6.goal_traces.max())

# Mean visit rates around trial 0, reading (a), blocks 1 to 8
vs = visit_sets(session_set("a: sessions 1 and 3 in order"))
tab = []
for s, (G, N) in vs.items():
    gm, nm, p = trialwise(G, N)
    for t in range(-3, 11):
        j = rt.VISIT_WINDOW + t
        tab.append({"subject": s, "trial": t, "goal": round(gm[j], 3), "nongoal": round(nm[j], 3), "p": f"{p[j]:.1e}",
                    "n_goal": int(np.sum(~np.isnan(G[:, j]))), "n_nongoal": int(np.sum(~np.isnan(N[:, j])))})
tab = pd.DataFrame(tab)
print("\nVisit probability of the goal arm and of never-goal arms, trials -3 to +10, reading (a):")
print(tab.pivot(index="trial", columns="subject", values=["goal", "nongoal", "p"]).reindex(columns=rt.SUBJECTS, level=1).to_string())
print("\nGoal traces per subject (legend of Figure 6A: 89, 85, 104 and 161 goal arms; 347, 437, 268 and 397 unrewarded arms):",
      {s: (len(G), len(N)) for s, (G, N) in vs.items()})

# Late significant trials: counts behind them, under both forms of the test
for s, (G, N) in vs.items():
    gm, nm, p = trialwise(G, N)
    for t in OFFS[(OFFS > 5) & (p < 0.05 / 81) & (gm > nm)]:
        j = rt.VISIT_WINDOW + t
        g, n = G[:, j][~np.isnan(G[:, j])], N[:, j][~np.isnan(N[:, j])]
        print(f"\n{s}, trial +{t}: goal visited in {int(g.sum())} of {len(g)} traces, never-goal arms in {int(n.sum())} of {len(n)}; "
              f"p = {p[j]:.1e} with tie correction, {stats.ranksums(g, n).pvalue:.3f} without")


**Result: blocked (too_large).** The criterion also needs the replay enrichment of
fig6a-pg-replay-persists, which needs ripples and decoding; only the behavioural half is computed
here, and it holds in three subjects of four. On trial +1, the closing trial of the block, the goal arm
is visited in 0.84 to 0.90 of blocks, then in 0.33 to 0.48 on +2 and 0.23 to 0.26 on +3, against 0.06
to 0.12 for the never-goal arms. In jaq, despereaux and montague the last trial after 0 with the goal
significantly above the never-goal arms is +2 or +3, within both bounds, at 0.05/80 and 0.05/81, under
every session reading, with completed blocks only, and for jaq over all 26 reading (b) candidates
(+2 to +3). In roquefort trials +1 to +3 are significant, and so is the single trial +29 (goal visited in
9 of 39 traces, never-goal arms in 8 of 170, p = 1.6e-4), so its last significant trial is +29 and the
behavioural half fails under both bounds. Without tie correction (scipy's `ranksums`), +29 is not
significant (p = 0.074) and every subject ends at +2, within both bounds. The claims file says only
"Wilcoxon rank-sum test", so the behavioural half's result depends on this choice, and a question is
asked. No subject has a trial after 0 with the goal significantly below the never-goal arms. The goal
traces are 94, 95, 112 and 169 under reading (a), against the 89, 85, 104 and 161 goal arms of the
legend.


## Claims not tested

The 31 other claims (fig6b-visits-decay among them, for its replay half) rest on sharp-wave ripples, multiunit events or the clusterless state space
decoder. The dandiset holds neither these nor the spikes or LFP they are computed from, only the
raw 30 kHz broadband, about 9.1 TB in total and stored one channel per chunk. The data check above
measures the streaming rate and shows that reading the run epochs once would take far longer than
the 30-minute limit, before any spike detection, ripple filtering or decoding. They are blocked
as too_large. Ripple detection would also need the CA1 cell-layer tetrodes and the corpus
callosum reference tetrode, which the electrode table does not identify (every location is
"None"). fig6b-visits-decay is behavioural in part: its behavioural half is computed in the section above,
but its criterion compares the visit decay with the replay enrichment of fig6a-pg-replay-persists,
so it is blocked with it.

## Summary

| claim | outcome |
|---|---|
| fig1d-sessions-blocks | reproduced |
| fig1ef-reward-rate | reproduced |
| fig6b-visits-decay | blocked, too_large (behavioural half: holds in jaq, despereaux and montague; fails in roquefort at trial +29 with tie correction, holds without) |
| the 30 other replay, ripple and decoding claims | blocked, too_large |

How the analysis differs from the paper's. Trials are rebuilt from the StateScript logs in the
files, not from the paper's preprocessed trial structures. Task phases follow the claims file's
account of label_trial_interval.m, with the first valid trial after each goal change in the old
block, as a reader's answer settled from the code. Which jaq sessions are "sessions 1 and 3" is
not fixed by the paper. Both readings the criterion allows are run: (a) in order of recording,
and (b) the sessions the Methods describe. Under (b), day 6 epoch 4 is the one session where not
all arms were visited, and the session with poor decoding cannot be found without the decoder, so
every candidate is run. All readings give the same outcomes. Sessions are not selected by decode
quality (the code's gooddecode), which needs the decoder. The rank-sum test of fig6b-visits-decay
is run with and without tie correction, since the claims file does not say which.

What was found about the dataset.
- The files hold raw broadband, behavioural events, position and task logs, but no spikes, LFP,
  ripple times, linearised position or decoding results. Every replay finding of the paper has to
  be recomputed from about 9.1 TB of raw data, which this registry cannot do.
- The electrode table gives no location for any of the 128 channels, so the CA1 tetrodes and the
  reference tetrode cannot be identified.
- The files do not mark which run sessions the paper analysed. They hold 27 and 24 run epochs
  for jaq and despereaux, where the paper used 26 (before excluding two) and 23. Which epoch is
  extra cannot be told; leaving out any one keeps the block counts within tolerance.
- No log uses forage assist, including the five without a header. Two blocks have one reward
  fewer than programmed because the first trial after the switch, a rewarded visit to the new
  goal, belongs to the old block.
- Trial counts are close to the legends' for search and correct repeat trials, but error trials
  are 11 to 24% more than Figure 5B's.
- The behaviour in the logs matches the hardware: rewards equal the pump onsets in 115 of 116
  epochs. The block counts are within 5% of the paper's for jaq (88 to 93 against 89) and
  despereaux (109 against 105), and equal for roquefort (85) and montague (166).
- The draft version's files were rewritten in 2022, and in the file compared its position
  timestamps are off the DIO clock by about -1.59e9 s; the published version is consistent.
- Position is sampled at about 33 Hz (Methods: 30 Hz); most epochs end with NaN timestamps.
- In one epoch (montague day 16 epoch 2) the log's clock steps by about 3.4 s against the DIO,
  and in one (jaq day 6 epoch 4) the log stops before a final pump.